In [1]:
from dbfread import DBF
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
file_path = "../dataset/CAMS AMCs NAVs 09052025/11052025150310_185847860R1.dbf"

output_dir = Path("../dataset/CAMS AMCs NAVs 09052025/processed")
products_dir = output_dir / "products"

products_dir.mkdir(parents=True, exist_ok=True)

In [3]:
dbf = DBF(file_path)

In [4]:
# Constant Columns
metadata_cols = [
    "PRODCODE",
    "NAME",
    "SCHEME_TYP",
    "ISIN_NO"
]

data_cols = [
    "NAV_DATE",
    "NAV_VALUE",
    "DIV_PERUNI",
    "CORPDIVRAT",
    "SWING_NAV"
]

In [5]:
metadata = []

current_code = None
current_metadata = None
buffer = []

In [6]:
# Process the DBF record by record
for record in dbf:

    code = record["PRODCODE"]

    # First product
    if current_code is None:
        current_code = code

        current_metadata = {
            col: record[col]
            for col in metadata_cols
        }

    # New product encountered
    if code != current_code:

        # Write previous product
        table = pa.Table.from_pylist(buffer)
        pq.write_table(
            table,
            products_dir / f"{current_code}.parquet"
        )

        # Save metadata
        metadata.append(current_metadata)

        # Reset for new product
        buffer = []
        current_code = code

        current_metadata = {
            col: record[col]
            for col in metadata_cols
        }

    # Store only variable/time-series columns
    buffer.append({
        col: record[col]
        for col in data_cols
    })

In [7]:
# Write the final product
if buffer:

    table = pa.Table.from_pylist(buffer)

    pq.write_table(
        table,
        products_dir / f"{current_code}.parquet"
    )

    metadata.append(current_metadata)

In [8]:
# Write metadata
metadata_table = pa.Table.from_pylist(metadata)

pq.write_table(
    metadata_table,
    output_dir / "product_metadata.parquet"
)

In [9]:
print("Extraction complete.")
print(f"Products: {len(metadata)}")
print(f"Output: {output_dir}")

Extraction complete.
Products: 26345
Output: ..\dataset\CAMS AMCs NAVs 09052025\processed
